# 🚕 MobilityFlow — smart-mobility data pipeline on free Colab
End-to-end ELT: **NYC TLC taxi trips + Open-Meteo weather → DuckDB** with bronze / silver / gold layers,
a data-quality gate, incremental loads and analytics marts.

Runtime: CPU is enough. Three months of green-taxi data takes a couple of minutes and well under 1 GB RAM.

## 1. Get the code

In [ ]:
!git clone -q https://github.com/sm25mtech11008-beep/mobilityflow.git
%cd mobilityflow
!pip install -q duckdb

## 2. Run the pipeline
Real data from NYC TLC + Open-Meteo. Swap `--source nyc` for `--source synthetic` if the download servers are unreachable.

In [ ]:
!python -m mobilityflow.pipeline --source nyc --start 2024-01 --end 2024-03

### Incremental re-run
Adding one more month only processes the new one — the first three are skipped via the watermark table.

In [ ]:
!python -m mobilityflow.pipeline --source nyc --start 2024-01 --end 2024-04

## 3. Query the warehouse with SQL

In [ ]:
import duckdb
con = duckdb.connect("data/warehouse.duckdb", read_only=True)
con.sql("SELECT table_name FROM information_schema.tables ORDER BY 1").show()

In [ ]:
con.sql("""
SELECT * FROM mart_weather_impact ORDER BY trips_per_hour DESC
""").df()

In [ ]:
con.sql("""
SELECT borough, SUM(trips) trips, ROUND(SUM(est_co2_tonnes),1) co2_t,
       ROUND(AVG(pct_short_trips_under_1_5mi),1) pct_short
FROM mart_borough_emissions GROUP BY borough ORDER BY co2_t DESC
""").df()

In [ ]:
con.sql("SELECT * FROM mart_congestion_hotspots ORDER BY avg_peak_speed_mph LIMIT 10").df()

## 4. Observability: data quality & run history

In [ ]:
con.sql("SELECT reject_reason, COUNT(*) n FROM silver_trips_rejected GROUP BY 1 ORDER BY n DESC").df()

In [ ]:
con.sql("SELECT * FROM dq_results ORDER BY run_id DESC, source_month LIMIT 12").df()

In [ ]:
con.sql("SELECT run_id, status, months_processed, clean_rows, rejected_rows, timings FROM pipeline_runs ORDER BY started_at").df()

## 5. View the report

In [ ]:
from IPython.display import HTML
con.close()
HTML(open("reports/report.html").read())

## 6. Run the tests

In [ ]:
!pip install -q pytest && python -m pytest -q